# Custom YOLO11s Training: Dog/Cat Detection → cvimodel

**Pipeline:** Upload VOC dataset → Train YOLO11s → Export ONNX → Convert cvimodel (Method B) → Deploy to MaixCAM

- **Dataset:** 200 images (100 cat + 100 dog), VOC format
- **Model:** YOLO11s (pre-trained COCO, fine-tuned)
- **Input size:** 224x320
- **Output:** cvimodel INT8 สำหรับ cv181x

In [1]:
# Step 1: Install condacolab
!pip install -q condacolab
import condacolab
condacolab.install()

⏬ Downloading https://github.com/jaimergp/miniforge/releases/download/24.11.2-1_colab/Miniforge3-colab-24.11.2-1_colab-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:08
🔁 Restarting kernel...


In [1]:
# Step 2: Clone server repo + setup conda env + tpu-mlir
!git clone https://github.com/KidBrightAI/kidbright_MAI_server server
%cd server
!tar -xf tools/opencv-3.4.13.tar.gz -C /
!chmod +x tools/onnx2ncnn
!chmod +x tools/spnntools
!conda create -n kbmai python=3.10 -y
!wget -q https://github.com/sophgo/tpu-mlir/releases/download/v1.27/tpu_mlir-1.27-py3-none-any.whl

Cloning into 'server'...
remote: Enumerating objects: 2039, done.
remote: Counting objects: 100% (409/409), done.
remote: Compressing objects: 100% (212/212), done.
remote: Total 2039 (delta 195), reused 407 (delta 195), pack-reused 1630 (from 2)
Receiving objects: 100% (2039/2039), 253.51 MiB | 17.19 MiB/s, done.
Resolving deltas: 100% (893/893), done.
Updating files: 100% (1590/1590), done.
/content/server
Channels:
 - conda-forge
Platform: linux-64
Solving environment: | / - done


==> WARNING: A newer version of conda exists. <==
    current version: 24.11.2
    latest version: 26.3.1

Please update conda by running

    $ conda update -n base -c conda-forge conda



## Package Plan ##

  environment location: /usr/local/envs/kbmai

  added / updated specs:
    - python=3.10


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    _openmp_mutex-4.5          |           20_gnu         

In [2]:
%%bash
source activate kbmai
pip install -q tpu_mlir*.whl

Reason for being yanked: deprecated, use 4.8.0.76


In [3]:
# Step 3: Install Python dependencies
!conda run -n kbmai pip -q install "setuptools<70.0.0" psutil flatbuffers "onnx==1.14.1" "onnxruntime==1.16.3" "onnxruntime_extensions==0.14.0" "torch==2.1.0" protobuf flask torchvision torchsummary
!conda run -n kbmai pip -q install "onnxsim==0.4.17" "onnxslim>=0.1.71" onnxruntime-gpu
!conda run -n kbmai pip -q install ultralytics "numpy<2" matplotlib-inline

## Upload Dataset

อัพโหลด dataset จาก `experiment/dataset/` (VOC format)  
โครงสร้าง: `JPEGImages/*.jpg` + `Annotations/*.xml` + `ImageSets/Main/train.txt`

In [4]:
# Step 4: Download dataset from repo (already committed)
import os, shutil

DATASET_SRC = "/content/server/experiment/dataset"
PROJECT_DIR = "/content/server/project_dogcat"
DATASET_DST = os.path.join(PROJECT_DIR, "dataset")

os.makedirs(PROJECT_DIR, exist_ok=True)
if os.path.exists(DATASET_DST):
    shutil.rmtree(DATASET_DST)
shutil.copytree(DATASET_SRC, DATASET_DST)

# Verify
for d in ["JPEGImages", "Annotations", "ImageSets/Main"]:
    path = os.path.join(DATASET_DST, d)
    count = len(os.listdir(path)) if os.path.isdir(path) else 0
    print(f"  {d}: {count} files")

# Show labels from annotations
from xml.etree import ElementTree as ET
labels = set()
for f in os.listdir(os.path.join(DATASET_DST, "Annotations")):
    tree = ET.parse(os.path.join(DATASET_DST, "Annotations", f))
    for obj in tree.findall(".//object"):
        labels.add(obj.find("name").text)
labels = sorted(labels)
print(f"\nLabels found: {labels}")
print(f"Dataset ready at: {DATASET_DST}")

  JPEGImages: 200 files
  Annotations: 200 files
  ImageSets/Main: 1 files

Labels found: ['cat', 'dog']
Dataset ready at: /content/server/project_dogcat/dataset


## Train YOLO11s

Fine-tune YOLO11s pre-trained (COCO) ด้วย custom dog/cat dataset  
Pipeline เดียวกับ `train_object_detection_yolo11n.py`: VOC → YOLO format → ultralytics train

In [5]:
# Step 5: Convert VOC to YOLO format + Train
import os, shutil, random, cv2, yaml
from xml.etree import ElementTree as ET

PROJECT_DIR = "/content/server/project_dogcat"
LABELS = ["cat", "dog"]
TRAIN_SPLIT = 80  # 80% train, 20% val
EPOCHS = 50
BATCH_SIZE = 16
LR = 0.001
IMGSZ = [224, 320]  # height, width

# --- Convert VOC to YOLO format ---
dataset_dir = os.path.join(PROJECT_DIR, "dataset")
yolo_dir = os.path.join(PROJECT_DIR, "yolo_dataset")

if os.path.exists(yolo_dir):
    shutil.rmtree(yolo_dir)

for split in ["train", "val"]:
    os.makedirs(os.path.join(yolo_dir, "images", split), exist_ok=True)
    os.makedirs(os.path.join(yolo_dir, "labels", split), exist_ok=True)

# Get image IDs from train.txt
train_txt = os.path.join(dataset_dir, "ImageSets", "Main", "train.txt")
with open(train_txt) as f:
    img_ids = [line.strip() for line in f if line.strip()]

random.seed(42)
random.shuffle(img_ids)
split_idx = int(len(img_ids) * (TRAIN_SPLIT / 100.0))
train_ids = img_ids[:split_idx]
val_ids = img_ids[split_idx:]
if not val_ids:
    val_ids = train_ids.copy()

print(f"Total: {len(img_ids)}, Train: {len(train_ids)}, Val: {len(val_ids)}")

def process_split(ids, split_name):
    count = 0
    for img_id in ids:
        img_path = os.path.join(dataset_dir, "JPEGImages", f"{img_id}.jpg")
        xml_path = os.path.join(dataset_dir, "Annotations", f"{img_id}.xml")
        if not os.path.exists(img_path) or not os.path.exists(xml_path):
            continue
        img = cv2.imread(img_path)
        if img is None:
            continue
        h, w = img.shape[:2]

        tree = ET.parse(xml_path)
        bboxes = []
        for obj in tree.findall(".//object"):
            name = obj.find("name").text.lower().strip()
            if name not in LABELS:
                continue
            cls_id = LABELS.index(name)
            bb = obj.find("bndbox")
            xmin, xmax = float(bb.find("xmin").text), float(bb.find("xmax").text)
            ymin, ymax = float(bb.find("ymin").text), float(bb.find("ymax").text)
            x_center = ((xmin + xmax) / 2.0) / w
            y_center = ((ymin + ymax) / 2.0) / h
            bw = (xmax - xmin) / w
            bh = (ymax - ymin) / h
            bboxes.append(f"{cls_id} {x_center} {y_center} {bw} {bh}")

        shutil.copy(img_path, os.path.join(yolo_dir, "images", split_name, f"{img_id}.jpg"))
        with open(os.path.join(yolo_dir, "labels", split_name, f"{img_id}.txt"), "w") as f:
            f.write("\n".join(bboxes))
        count += 1
    return count

t = process_split(train_ids, "train")
v = process_split(val_ids, "val")
print(f"Converted: train={t}, val={v}")

# Generate data.yaml
data_yaml = {
    "path": os.path.abspath(yolo_dir),
    "train": "images/train",
    "val": "images/val",
    "names": {i: name for i, name in enumerate(LABELS)}
}
yaml_path = os.path.join(yolo_dir, "data.yaml")
with open(yaml_path, "w") as f:
    yaml.dump(data_yaml, f, default_flow_style=False)
print(f"\ndata.yaml: {yaml_path}")
!cat {yaml_path}

Total: 200, Train: 160, Val: 40
Converted: train=160, val=40

data.yaml: /content/server/project_dogcat/yolo_dataset/data.yaml
names:
  0: cat
  1: dog
path: /content/server/project_dogcat/yolo_dataset
train: images/train
val: images/val


In [7]:
# Step 6: Train YOLO11s (50 epochs, GPU)
!conda run -n kbmai --live-stream python -c "from ultralytics import YOLO; model = YOLO('yolo11s.pt'); model.train(data='/content/server/project_dogcat/yolo_dataset/data.yaml', epochs=50, batch=16, imgsz=[224, 320], device=0, lr0=0.001, project='/content/server/project_dogcat/output', name='yolo11s_dogcat', exist_ok=True, verbose=True); print('Training done!')"

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.36 🚀 Python-3.10.20 torch-2.1.0+cu121 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/server/project_dogcat/yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v

In [8]:
# Step 7: Check training results
import os
best_pt = "/content/server/project_dogcat/output/yolo11s_dogcat/weights/best.pt"
last_pt = "/content/server/project_dogcat/output/yolo11s_dogcat/weights/last.pt"
print(f"best.pt exists: {os.path.exists(best_pt)} ({os.path.getsize(best_pt)/1e6:.1f} MB)" if os.path.exists(best_pt) else "best.pt NOT FOUND")
print(f"last.pt exists: {os.path.exists(last_pt)}" if os.path.exists(last_pt) else "")

# Show results
results_csv = "/content/server/project_dogcat/output/yolo11s_dogcat/results.csv"
if os.path.exists(results_csv):
    import pandas as pd
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()
    print(f"\nTraining Results (last 5 epochs):")
    print(df[["epoch", "metrics/mAP50(B)", "metrics/mAP50-95(B)", "train/box_loss", "train/cls_loss"]].tail())

best.pt exists: True (19.1 MB)
last.pt exists: True

Training Results (last 5 epochs):
    epoch  metrics/mAP50(B)  metrics/mAP50-95(B)  train/box_loss  \
45     46           0.96435              0.77593         0.44203   
46     47           0.97709              0.79655         0.44387   
47     48           0.98197              0.78984         0.45808   
48     49           0.98673              0.79679         0.43701   
49     50           0.98840              0.80418         0.43375   

    train/cls_loss  
45         0.33298  
46         0.33865  
47         0.34933  
48         0.32215  
49         0.32843  


## Export ONNX + Convert to cvimodel

Export best.pt → ONNX → MLIR → INT8 cvimodel (Method B, 2 output nodes)

In [9]:
# Step 8: Export ONNX from best.pt
!conda run -n kbmai --live-stream python -c "from ultralytics import YOLO; model = YOLO('/content/server/project_dogcat/output/yolo11s_dogcat/weights/best.pt'); model.export(format='onnx', imgsz=[224, 320], opset=16); print('ONNX export done!')"
!ls -lh /content/server/project_dogcat/output/yolo11s_dogcat/weights/best.onnx

Ultralytics 8.4.36 🚀 Python-3.10.20 torch-2.1.0+cu121 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLO11s summary (fused): 101 layers, 9,413,574 parameters, 0 gradients, 21.3 GFLOPs

PyTorch: starting from '/content/server/project_dogcat/output/yolo11s_dogcat/weights/best.pt' with input shape (1, 3, 224, 320) BCHW and output shape(s) (1, 6, 1470) (18.3 MB)

ONNX: starting export with onnx 1.14.1 opset 16...
ONNX: slimming with onnxslim 0.1.90...
ONNX: export success ✅ 2.0s, saved as '/content/server/project_dogcat/output/yolo11s_dogcat/weights/best.onnx' (36.0 MB)

Export complete (2.4s)
Results saved to /content/server/project_dogcat/output/yolo11s_dogcat/weights
Predict:         yolo predict task=detect model=/content/server/project_dogcat/output/yolo11s_dogcat/weights/best.onnx imgsz=224,320 
Validate:        yolo val task=detect model=/content/server/proj

In [10]:
# Step 9: Verify ONNX output nodes
!conda run -n kbmai --live-stream python -c "import onnx; model = onnx.load('/content/server/project_dogcat/output/yolo11s_dogcat/weights/best.onnx'); print('Output nodes:'); [print(f'  {o.name}: {[d.dim_value for d in o.type.tensor_type.shape.dim]}') for o in model.graph.output]; targets = ['/model.23/dfl/conv/Conv_output_0', '/model.23/Sigmoid_output_0']; all_out = {o for n in model.graph.node for o in n.output}; [print(f'  {t}: FOUND' if t in all_out else f'  {t}: NOT FOUND') for t in targets]"

Output nodes:
  output0: [1, 6, 1470]
  /model.23/dfl/conv/Conv_output_0: FOUND
  /model.23/Sigmoid_output_0: FOUND


In [11]:
# Step 10: model_transform (Method B - 2 output nodes)
ONNX_PATH = "/content/server/project_dogcat/output/yolo11s_dogcat/weights/best.onnx"

!conda run -n kbmai model_transform.py \
  --model_name yolo11s_dogcat \
  --model_def {ONNX_PATH} \
  --input_shapes [[1,3,224,320]] \
  --mean "0,0,0" \
  --output_names "/model.23/dfl/conv/Conv_output_0,/model.23/Sigmoid_output_0" \
  --scale "0.00392156862745098,0.00392156862745098,0.00392156862745098" \
  --keep_aspect_ratio \
  --pixel_format rgb \
  --channel_format nchw \
  --test_input /content/server/data/test_images2/cat.jpg \
  --test_result yolo11s_dogcat_top_outputs.npz \
  --tolerance 0.99,0.99 \
  --mlir yolo11s_dogcat.mlir

2026/04/09 09:23:08 - INFO : TPU-MLIR v1.27-20260206
2026/04/09 09:23:08 - INFO : 
	 _____________________________________________________ 
	| preprocess:                                           |
	|   (x - mean) * scale                                  |
	'-------------------------------------------------------'
  config Preprocess args : 
	resize_dims           : same to net input dims
	keep_aspect_ratio     : True
	keep_ratio_mode       : letterbox
	pad_value             : 0
	pad_type              : center
	--------------------------
	mean                  : [0.0, 0.0, 0.0]
	scale                 : [0.003921569, 0.003921569, 0.003921569]
	--------------------------
	pixel_format          : rgb
	channel_format        : nchw
	yuv_type              : 

2026/04/09 09:23:12 - INFO : Input_shape assigned
2026/04/09 09:23:12 - INFO : ConstantFolding finished
2026/04/09 09:23:12 - INFO : skip_fuse_bn:False
2026/04/09 09:23:13 - INFO : Onnxsim opt finished
2026/04/09 09:23:13 - INFO : Cons

In [12]:
# Step 11: Calibration (use training images for better quantization)
!conda run -n kbmai run_calibration.py yolo11s_dogcat.mlir \
  --dataset /content/server/project_dogcat/yolo_dataset/images/train \
  --input_num 24 \
  -o yolo11s_dogcat_cali_table

TPU-MLIR v1.27-20260206
2026/04/09 09:23:41 - INFO : 
  load_config Preprocess args : 
	resize_dims           : [224, 320]
	keep_aspect_ratio     : True
	keep_ratio_mode       : letterbox
	pad_value             : 0
	pad_type              : center
	input_dims            : [224, 320]
	--------------------------
	mean                  : [0.0, 0.0, 0.0]
	scale                 : [0.003921569, 0.003921569, 0.003921569]
	--------------------------
	pixel_format          : rgb
	channel_format        : nchw
	yuv_type              : 

input_num = 24, ref = 24
real input_num = 24
auto tune end, run time:141.8876268863678


activation_collect_and_calc_th for sample: 0:   0%|          | 0/24 [00:00<?, ?it/s][                                                  ] 0%                                                  ] 0%                                                  ] 0%

In [13]:
# Step 12: Deploy to cvimodel (INT8, cv181x)
!conda run -n kbmai model_deploy.py \
  --mlir yolo11s_dogcat.mlir \
  --quantize INT8 \
  --calibration_table yolo11s_dogcat_cali_table \
  --processor cv181x \
  --test_input yolo11s_dogcat_in_f32.npz \
  --test_reference yolo11s_dogcat_top_outputs.npz \
  --tolerance 0.85,0.5 \
  --model yolo11s_dogcat.cvimodel

2026/04/09 09:26:47 - INFO : TPU-MLIR v1.27-20260206


[/model.23/dfl/conv/Conv_output_0_Conv]      SIMILAR [PASSED]
    (1, 1, 4, 1470) float32 
    cosine_similarity      = 0.996226
    euclidean_similarity   = 0.901252
    sqnr_similarity        = 13.953633
[/model.23/Sigmoid_output_0_Sigmoid]      SIMILAR [PASSED]
    (1, 2, 1470) float32 
    cosine_similarity      = 0.998000
    euclidean_similarity   = 0.926330
    sqnr_similarity        = 22.485824
2 compared
2 passed
  0 equal, 0 close, 2 similar
0 failed
  0 not equal, 0 not similar
min_similiarity = (0.9962261915206909, 0.9012517011566945, 13.953633308410645)
Target    yolo11s_dogcat_cv181x_int8_sym_tpu_outputs.npz
Reference yolo11s_dogcat_top_outputs.npz
npz compare PASSED.
Dumping LgConfig!
Global Configs:
  shape_secs_search_strategy = 0
  structure_detect_opt = 1
Search Method Config: sc_method_quick_search
  CSECS_SEARCH_RECORD_THRESHOLD = -1
  DSECS_SEARCH_RECORD_THRESHOLD = -1
  HSECS_SEARCH_RECORD_THRESHOLD = -1
  MA

In [14]:
# Step 13: Create MUD file + Check result
mud_content = """[basic]
type = cvimodel
model = yolo11s_dogcat.cvimodel

[extra]
model_type = yolo11
input_type = rgb
mean = 0, 0, 0
scale = 0.00392156862745098, 0.00392156862745098, 0.00392156862745098
labels = cat, dog
"""
with open("yolo11s_dogcat.mud", "w") as f:
    f.write(mud_content)

import os
cvi = "yolo11s_dogcat.cvimodel"
if os.path.exists(cvi):
    size_mb = os.path.getsize(cvi) / 1e6
    print(f"SUCCESS! {cvi}: {size_mb:.1f} MB")
    print(f"MUD file: yolo11s_dogcat.mud")
else:
    print("FAILED: cvimodel not generated")

SUCCESS! yolo11s_dogcat.cvimodel: 10.0 MB
MUD file: yolo11s_dogcat.mud


In [15]:
# Step 14: Download cvimodel + MUD
from google.colab import files
files.download("yolo11s_dogcat.cvimodel")
files.download("yolo11s_dogcat.mud")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>